# 4. Provider Performance & Network Analytics
## Vertically Complex Provider Intelligence Pipeline

Strategy: LIMIT 1000 → Local CSV → Pandas → Spark

## Available Tables:
- **OMOP**: 24 tables (provider, care_site, procedure_occurrence, drug_exposure, condition_occurrence, observation, person, death)
- **Medicare**: 6 tables (physicians_and_other_supplier)
- **NPPES**: 1 table (healthcare_provider_taxonomy_code_set)

## Pipeline Architecture:
- **Bronze**: 10 tables (raw data)
- **Silver**: 6 intermediate layers (provider profiling)
- **Gold**: 15 vertical layers → 4 final metrics

## Final Metrics:
1. **Provider Quality Composite** - Quality indicator integration
2. **Network Efficiency Score** - Referral network efficiency
3. **Patient Outcome Attribution** - Provider-specific outcome contribution
4. **Care Coordination Index** - Collaborative care quality

In [1]:
from google.cloud import bigquery
import pandas as pd
from pyspark.sql import SparkSession, functions as F, Window as W
from pyspark.sql.types import *
import os
from datetime import datetime
import networkx as nx
import json

In [2]:
print("Initializing Spark...")
spark = SparkSession.builder \
    .appName("CMS_ProviderPerformance") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.maxResultSize", "2g") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

Initializing Spark...


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/02 22:57:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/12/02 22:57:06 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/12/02 22:57:06 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
25/12/02 22:57:06 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
25/12/02 22:57:06 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.


Spark version: 3.5.1
Spark UI: http://mac:4044


In [3]:
PROJECT_ID = "opportune-ruler-447319-b3"
LIMIT = 1000
LOCAL_DATA_DIR = "./4_data"

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

DATASETS = {
    "omop": "bigquery-public-data.cms_synthetic_patient_data_omop",
    "medicare": "bigquery-public-data.cms_medicare",
    "nppes": "bigquery-public-data.nppes"
}

print(f"Config:")
print(f"  Project: {PROJECT_ID}")
print(f"  Limit: {LIMIT} rows per table")
print(f"  Data dir: {LOCAL_DATA_DIR}")

Config:
  Project: opportune-ruler-447319-b3
  Limit: 1000 rows per table
  Data dir: ./4_data


# STEP 1: Download from BigQuery (LIMIT 1000)

In [13]:
def download_table(client, dataset_key, table_name, limit=1000):
    """Download table from BigQuery to local CSV"""
    try:
        source = DATASETS[dataset_key]
        query = f"SELECT * FROM `{source}.{table_name}` LIMIT {limit}"
        df = client.query(query).to_dataframe()
        
        output_file = f"{LOCAL_DATA_DIR}/{dataset_key}_{table_name}.csv"
        df.to_csv(output_file, index=False)
        print(f"  ✓ {dataset_key}.{table_name}: {len(df)} rows → {output_file}")
        return True
    except Exception as e:
        print(f"  ✗ {dataset_key}.{table_name}: {str(e)}")
        return False

In [14]:
print("\n" + "="*60)
print("DOWNLOADING TABLES")
print("="*60)

client = bigquery.Client(project=PROJECT_ID)

tables_to_download = [
    ("omop", "provider"),
    ("omop", "care_site"),
    ("omop", "person"),
    ("omop", "death"),
    ("omop", "procedure_occurrence"),
    ("omop", "drug_exposure"),
    ("omop", "condition_occurrence"),
    ("omop", "observation"),
    ("medicare", "physicians_and_other_supplier_2014"),
    ("nppes", "healthcare_provider_taxonomy_code_set")
]

for dataset_key, table_name in tables_to_download:
    download_table(client, dataset_key, table_name, LIMIT)

print(f"\n✓ Downloaded {len(tables_to_download)} tables")


DOWNLOADING TABLES


/opt/anaconda3/envs/moicaprice/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


  ✓ omop.provider: 1000 rows → ./4_data/omop_provider.csv


/opt/anaconda3/envs/moicaprice/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


  ✓ omop.care_site: 1000 rows → ./4_data/omop_care_site.csv


...

  ✓ omop.observation: 1000 rows → ./4_data/omop_observation.csv


/opt/anaconda3/envs/moicaprice/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


  ✓ medicare.physicians_and_other_supplier_2014: 1000 rows → ./4_data/medicare_physicians_and_other_supplier_2014.csv


/opt/anaconda3/envs/moicaprice/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


  ✓ nppes.healthcare_provider_taxonomy_code_set: 853 rows → ./4_data/nppes_healthcare_provider_taxonomy_code_set.csv

✓ Downloaded 10 tables


# STEP 2: Load CSV → Pandas → Spark (Bronze Layer)

In [4]:
def load_csv_to_spark(file_name):
    """Load CSV via Pandas then convert to Spark DataFrame"""
    try:
        csv_path = f"{LOCAL_DATA_DIR}/{file_name}"
        if not os.path.exists(csv_path):
            print(f"  ✗ File not found: {csv_path}")
            return None
        
        pandas_df = pd.read_csv(csv_path)
        spark_df = spark.createDataFrame(pandas_df)
        print(f"  ✓ Loaded {file_name}: {spark_df.count()} rows, {len(spark_df.columns)} cols")
        return spark_df
    except Exception as e:
        print(f"  ✗ Error loading {file_name}: {str(e)}")
        return None

In [5]:
print("\n" + "="*60)
print("LOADING BRONZE LAYER")
print("="*60)

bronze_provider = load_csv_to_spark("omop_provider.csv")
bronze_care_site = load_csv_to_spark("omop_care_site.csv")
bronze_person = load_csv_to_spark("omop_person.csv")
bronze_death = load_csv_to_spark("omop_death.csv")
bronze_procedure = load_csv_to_spark("omop_procedure_occurrence.csv")
bronze_drug = load_csv_to_spark("omop_drug_exposure.csv")
bronze_condition = load_csv_to_spark("omop_condition_occurrence.csv")
bronze_observation = load_csv_to_spark("omop_observation.csv")
bronze_medicare = load_csv_to_spark("medicare_physicians_and_other_supplier_2014.csv")
bronze_taxonomy = load_csv_to_spark("nppes_healthcare_provider_taxonomy_code_set.csv")

print("\n✓ Bronze layer loaded (10 tables)")


LOADING BRONZE LAYER


  ✓ Loaded omop_provider.csv: 1000 rows, 13 cols
  ✓ Loaded omop_care_site.csv: 1000 rows, 6 cols
  ✓ Loaded omop_person.csv: 1000 rows, 18 cols
  ✓ Loaded omop_death.csv: 1000 rows, 7 cols
  ✓ Loaded omop_procedure_occurrence.csv: 1000 rows, 14 cols
  ✓ Loaded omop_drug_exposure.csv: 1000 rows, 23 cols
  ✓ Loaded omop_condition_occurrence.csv: 1000 rows, 16 cols
  ✓ Loaded omop_observation.csv: 1000 rows, 18 cols


25/12/02 22:57:43 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


  ✓ Loaded medicare_physicians_and_other_supplier_2014.csv: 1000 rows, 26 cols
  ✓ Loaded nppes_healthcare_provider_taxonomy_code_set.csv: 853 rows, 6 cols

✓ Bronze layer loaded (10 tables)


# STEP 3: Build Silver Layer (6 Intermediate Tables)

In [6]:
print("\n" + "="*60)
print("SILVER 1: Provider Demographics Enriched")
print("="*60)

silver_provider_demographics = bronze_provider \
    .join(bronze_care_site, bronze_provider.care_site_id == bronze_care_site.care_site_id, "left") \
    .select(
        bronze_provider.provider_id,
        bronze_provider.provider_name,
        bronze_provider.npi,
        bronze_provider.dea,
        bronze_provider.specialty_concept_id,
        bronze_provider.specialty_source_value,
        bronze_provider.care_site_id,
        bronze_provider.year_of_birth,
        bronze_provider.gender_concept_id,
        bronze_care_site.care_site_name,
        bronze_care_site.place_of_service_concept_id,
        F.lit(datetime.now().isoformat()).alias("processed_timestamp")
    )

print(f"✓ Provider demographics: {silver_provider_demographics.count()} providers")
silver_provider_demographics.show(5, truncate=False)


SILVER 1: Provider Demographics Enriched
✓ Provider demographics: 1000 providers
+-----------+-------------+----------+---+--------------------+----------------------+------------+-------------+-----------------+--------------+---------------------------+--------------------------+
|provider_id|provider_name|npi       |dea|specialty_concept_id|specialty_source_value|care_site_id|year_of_birth|gender_concept_id|care_site_name|place_of_service_concept_id|processed_timestamp       |
+-----------+-------------+----------+---+--------------------+----------------------+------------+-------------+-----------------+--------------+---------------------------+--------------------------+
|136290     |NaN          |8665861430|NaN|NaN                 |NaN                   |65792       |NaN          |NaN              |NULL          |NULL                       |2025-12-02T22:57:46.242259|
|380        |NaN          |2689606218|NaN|NaN                 |NaN                   |256         |NaN        

In [7]:
print("\n" + "="*60)
print("SILVER 2: Provider Clinical Activity")
print("="*60)

procedure_activity = bronze_procedure \
    .groupBy("provider_id") \
    .agg(
        F.count("*").alias("procedure_count"),
        F.countDistinct("person_id").alias("unique_patients_procedures"),
        F.countDistinct("procedure_concept_id").alias("procedure_variety")
    )

drug_activity = bronze_drug \
    .groupBy("provider_id") \
    .agg(
        F.count("*").alias("drug_count"),
        F.countDistinct("person_id").alias("unique_patients_drugs"),
        F.countDistinct("drug_concept_id").alias("drug_variety")
    )

condition_activity = bronze_condition \
    .groupBy("provider_id") \
    .agg(
        F.count("*").alias("condition_count"),
        F.countDistinct("person_id").alias("unique_patients_conditions"),
        F.countDistinct("condition_concept_id").alias("condition_variety")
    )

silver_provider_activity = procedure_activity \
    .join(drug_activity, "provider_id", "full_outer") \
    .join(condition_activity, "provider_id", "full_outer") \
    .fillna(0) \
    .withColumn("total_encounters", 
                F.col("procedure_count") + F.col("drug_count") + F.col("condition_count")) \
    .withColumn("total_unique_patients",
                F.greatest(F.col("unique_patients_procedures"), 
                          F.col("unique_patients_drugs"),
                          F.col("unique_patients_conditions"))) \
    .withColumn("processed_timestamp", F.lit(datetime.now().isoformat()))

print(f"✓ Provider activity: {silver_provider_activity.count()} providers")
silver_provider_activity.show(5, truncate=False)


SILVER 2: Provider Clinical Activity
✓ Provider activity: 2407 providers
+-----------+---------------+--------------------------+-----------------+----------+---------------------+------------+---------------+--------------------------+-----------------+----------------+---------------------+--------------------------+
|provider_id|procedure_count|unique_patients_procedures|procedure_variety|drug_count|unique_patients_drugs|drug_variety|condition_count|unique_patients_conditions|condition_variety|total_encounters|total_unique_patients|processed_timestamp       |
+-----------+---------------+--------------------------+-----------------+----------+---------------------+------------+---------------+--------------------------+-----------------+----------------+---------------------+--------------------------+
|45.0       |0              |0                         |0                |1         |1                    |1           |0              |0                         |0                |1

In [8]:
print("\n" + "="*60)
print("SILVER 3: Patient Outcomes by Provider")
print("="*60)

patient_outcomes = bronze_person \
    .join(bronze_death, "person_id", "left") \
    .join(bronze_condition, "person_id", "left") \
    .groupBy(bronze_condition.provider_id) \
    .agg(
        F.count(bronze_person.person_id).alias("total_patients"),
        F.sum(F.when(bronze_death.person_id.isNotNull(), 1).otherwise(0)).alias("deceased_patients"),
        F.avg(F.datediff(bronze_condition.condition_end_date, bronze_condition.condition_start_date)).alias("avg_condition_duration_days"),
        F.countDistinct(bronze_condition.condition_concept_id).alias("condition_complexity")
    ) \
    .withColumn("mortality_rate", 
                F.when(F.col("total_patients") > 0,
                      F.col("deceased_patients") / F.col("total_patients")).otherwise(0)) \
    .withColumn("processed_timestamp", F.lit(datetime.now().isoformat()))

silver_patient_outcomes = patient_outcomes

print(f"✓ Patient outcomes: {silver_patient_outcomes.count()} providers")
silver_patient_outcomes.show(5, truncate=False)


SILVER 3: Patient Outcomes by Provider
✓ Patient outcomes: 2 providers
+-----------+--------------+-----------------+---------------------------+--------------------+--------------------+--------------------------+
|provider_id|total_patients|deceased_patients|avg_condition_duration_days|condition_complexity|mortality_rate      |processed_timestamp       |
+-----------+--------------+-----------------+---------------------------+--------------------+--------------------+--------------------------+
|NULL       |999           |1                |NULL                       |0                   |0.001001001001001001|2025-12-02T22:57:48.630023|
|9069.0     |1             |0                |3.0                        |1                   |0.0                 |2025-12-02T22:57:48.630023|
+-----------+--------------+-----------------+---------------------------+--------------------+--------------------+--------------------------+



In [9]:
print("\n" + "="*60)
print("SILVER 4: Provider Referral Network")
print("="*60)

procedure_referrals = bronze_procedure \
    .select("person_id", "provider_id", "procedure_dat") \
    .withColumnRenamed("provider_id", "provider_from")

drug_providers = bronze_drug \
    .select("person_id", "provider_id", "drug_exposure_start_date") \
    .withColumnRenamed("provider_id", "provider_to") \
    .withColumnRenamed("drug_exposure_start_date", "referral_date")

referral_network = procedure_referrals \
    .join(drug_providers, "person_id") \
    .filter(F.col("provider_from") != F.col("provider_to")) \
    .groupBy("provider_from", "provider_to") \
    .agg(
        F.count("*").alias("referral_count"),
        F.countDistinct("person_id").alias("unique_patients_referred")
    )

outbound_referrals = referral_network \
    .groupBy("provider_from") \
    .agg(
        F.count("*").alias("outbound_referral_count"),
        F.sum("referral_count").alias("total_patients_referred_out")
    )

inbound_referrals = referral_network \
    .groupBy("provider_to") \
    .agg(
        F.count("*").alias("inbound_referral_count"),
        F.sum("referral_count").alias("total_patients_referred_in")
    )

silver_referral_network = outbound_referrals \
    .withColumnRenamed("provider_from", "provider_id") \
    .join(inbound_referrals.withColumnRenamed("provider_to", "provider_id"), "provider_id", "full_outer") \
    .fillna(0) \
    .withColumn("network_centrality", 
                F.col("outbound_referral_count") + F.col("inbound_referral_count")) \
    .withColumn("processed_timestamp", F.lit(datetime.now().isoformat()))

print(f"✓ Referral network: {silver_referral_network.count()} providers")
silver_referral_network.show(5, truncate=False)


SILVER 4: Provider Referral Network
✓ Referral network: 6 providers
+-----------+-----------------------+---------------------------+----------------------+--------------------------+------------------+--------------------------+
|provider_id|outbound_referral_count|total_patients_referred_out|inbound_referral_count|total_patients_referred_in|network_centrality|processed_timestamp       |
+-----------+-----------------------+---------------------------+----------------------+--------------------------+------------------+--------------------------+
|2317.0     |0                      |0                          |1                     |1                         |1                 |2025-12-02T22:57:49.593906|
|2632.0     |1                      |1                          |0                     |0                         |1                 |2025-12-02T22:57:49.593906|
|11087.0    |0                      |0                          |1                     |1                         |1     

In [10]:
print("\n" + "="*60)
print("SILVER 5: Provider Quality Indicators")
print("="*60)

readmission_proxy = bronze_condition \
    .withColumn("next_visit", F.lead("condition_start_date").over(
        W.partitionBy("person_id", "provider_id").orderBy("condition_start_date"))) \
    .withColumn("days_to_next_visit", 
                F.datediff(F.col("next_visit"), F.col("condition_end_date"))) \
    .withColumn("is_readmission", 
                F.when((F.col("days_to_next_visit") >= 0) & (F.col("days_to_next_visit") <= 30), 1).otherwise(0))

quality_indicators = readmission_proxy \
    .groupBy("provider_id") \
    .agg(
        F.count("*").alias("total_episodes"),
        F.sum("is_readmission").alias("readmissions_30day"),
        F.avg("days_to_next_visit").alias("avg_days_between_visits"),
        F.avg(F.datediff(F.col("condition_end_date"), F.col("condition_start_date"))).alias("avg_episode_length")
    ) \
    .withColumn("readmission_rate", 
                F.when(F.col("total_episodes") > 0,
                      F.col("readmissions_30day") / F.col("total_episodes")).otherwise(0)) \
    .withColumn("processed_timestamp", F.lit(datetime.now().isoformat()))

silver_quality_indicators = quality_indicators

print(f"✓ Quality indicators: {silver_quality_indicators.count()} providers")
silver_quality_indicators.show(5, truncate=False)


SILVER 5: Provider Quality Indicators
✓ Quality indicators: 798 providers
+-----------+--------------+------------------+-----------------------+------------------+----------------+--------------------------+
|provider_id|total_episodes|readmissions_30day|avg_days_between_visits|avg_episode_length|readmission_rate|processed_timestamp       |
+-----------+--------------+------------------+-----------------------+------------------+----------------+--------------------------+
|394862.0   |1             |0                 |NULL                   |7.0               |0.0             |2025-12-02T22:57:50.365301|
|49683.0    |4             |0                 |NULL                   |5.5               |0.0             |2025-12-02T22:57:50.365301|
|30985.0    |2             |0                 |NULL                   |2.5               |0.0             |2025-12-02T22:57:50.365301|
|118945.0   |3             |0                 |NULL                   |20.666666666666668|0.0             |2025-12-

In [11]:
print("\n" + "="*60)
print("SILVER 6: Provider Care Coordination")
print("="*60)

patient_provider_count = bronze_condition \
    .select("person_id", "provider_id") \
    .union(bronze_procedure.select("person_id", "provider_id")) \
    .union(bronze_drug.select("person_id", "provider_id")) \
    .distinct() \
    .groupBy("person_id") \
    .agg(F.countDistinct("provider_id").alias("provider_count_per_patient"))

coordination_metrics = bronze_condition \
    .join(patient_provider_count, "person_id") \
    .groupBy(bronze_condition.provider_id) \
    .agg(
        F.avg("provider_count_per_patient").alias("avg_providers_per_patient"),
        F.max("provider_count_per_patient").alias("max_providers_per_patient"),
        F.countDistinct(bronze_condition.person_id).alias("patients_requiring_coordination")
    ) \
    .withColumn("coordination_complexity_score",
                F.col("avg_providers_per_patient") * F.col("patients_requiring_coordination") / 100) \
    .withColumn("processed_timestamp", F.lit(datetime.now().isoformat()))

silver_care_coordination = coordination_metrics

print(f"✓ Care coordination: {silver_care_coordination.count()} providers")
silver_care_coordination.show(5, truncate=False)


SILVER 6: Provider Care Coordination
✓ Care coordination: 798 providers
+-----------+-------------------------+-------------------------+-------------------------------+-----------------------------+--------------------------+
|provider_id|avg_providers_per_patient|max_providers_per_patient|patients_requiring_coordination|coordination_complexity_score|processed_timestamp       |
+-----------+-------------------------+-------------------------+-------------------------------+-----------------------------+--------------------------+
|57517.0    |1.0                      |1                        |1                              |0.01                         |2025-12-02T22:57:50.833179|
|30985.0    |1.0                      |1                        |2                              |0.02                         |2025-12-02T22:57:50.833179|
|239907.0   |1.0                      |1                        |1                              |0.01                         |2025-12-02T22:57:50.83317

# STEP 4: Build Gold Layer (15 Vertical Transformations)

In [12]:
print("\n" + "="*60)
print("GOLD 1: Provider Master Profile")
print("="*60)

gold_provider_master = silver_provider_demographics \
    .join(silver_provider_activity, "provider_id", "left") \
    .join(silver_patient_outcomes, "provider_id", "left") \
    .fillna(0) \
    .withColumn("provider_age", F.lit(2024) - F.col("year_of_birth")) \
    .withColumn("patient_per_encounter_ratio",
                F.when(F.col("total_encounters") > 0,
                      F.col("total_unique_patients") / F.col("total_encounters")).otherwise(0)) \
    .withColumn("processed_timestamp", F.lit(datetime.now().isoformat()))

print(f"✓ Provider master profile: {gold_provider_master.count()} providers")
gold_provider_master.show(5, truncate=False)


GOLD 1: Provider Master Profile
✓ Provider master profile: 1000 providers
+-----------+-------------+----------+---+--------------------+----------------------+------------+-------------+-----------------+--------------+---------------------------+--------------------------+---------------+--------------------------+-----------------+----------+---------------------+------------+---------------+--------------------------+-----------------+----------------+---------------------+--------------------------+--------------+-----------------+---------------------------+--------------------+--------------+--------------------------+------------+---------------------------+
|provider_id|provider_name|npi       |dea|specialty_concept_id|specialty_source_value|care_site_id|year_of_birth|gender_concept_id|care_site_name|place_of_service_concept_id|processed_timestamp       |procedure_count|unique_patients_procedures|procedure_variety|drug_count|unique_patients_drugs|drug_variety|condition_count|

In [13]:
print("\n" + "="*60)
print("GOLD 2: Clinical Volume Metrics")
print("="*60)

volume_window = W.orderBy(F.col("total_encounters").desc())

gold_volume_metrics = silver_provider_activity \
    .withColumn("volume_rank", F.row_number().over(volume_window)) \
    .withColumn("volume_percentile", F.percent_rank().over(volume_window)) \
    .withColumn("is_high_volume", F.when(F.col("volume_percentile") >= 0.75, 1).otherwise(0)) \
    .withColumn("procedure_ratio", 
                F.when(F.col("total_encounters") > 0,
                      F.col("procedure_count") / F.col("total_encounters")).otherwise(0)) \
    .withColumn("drug_ratio",
                F.when(F.col("total_encounters") > 0,
                      F.col("drug_count") / F.col("total_encounters")).otherwise(0)) \
    .withColumn("processed_timestamp", F.lit(datetime.now().isoformat()))

print(f"✓ Volume metrics: {gold_volume_metrics.count()} providers")
gold_volume_metrics.show(5, truncate=False)


GOLD 2: Clinical Volume Metrics
✓ Volume metrics: 2407 providers


25/12/02 22:57:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 2

+-----------+---------------+--------------------------+-----------------+----------+---------------------+------------+---------------+--------------------------+-----------------+----------------+---------------------+--------------------------+-----------+---------------------+--------------+-------------------+----------+
|provider_id|procedure_count|unique_patients_procedures|procedure_variety|drug_count|unique_patients_drugs|drug_variety|condition_count|unique_patients_conditions|condition_variety|total_encounters|total_unique_patients|processed_timestamp       |volume_rank|volume_percentile    |is_high_volume|procedure_ratio    |drug_ratio|
+-----------+---------------+--------------------------+-----------------+----------+---------------------+------------+---------------+--------------------------+-----------------+----------------+---------------------+--------------------------+-----------+---------------------+--------------+-------------------+----------+
|0.0        |12 

In [14]:
print("\n" + "="*60)
print("GOLD 3: Clinical Diversity Index")
print("="*60)

gold_clinical_diversity = silver_provider_activity \
    .withColumn("diversity_score",
                (F.col("procedure_variety") + F.col("drug_variety") + F.col("condition_variety")) / 3.0) \
    .withColumn("specialization_index",
                F.when(F.col("diversity_score") > 0,
                      1.0 / F.col("diversity_score")).otherwise(0)) \
    .withColumn("is_specialist", F.when(F.col("specialization_index") > 0.1, 1).otherwise(0)) \
    .withColumn("is_generalist", F.when(F.col("diversity_score") > 50, 1).otherwise(0)) \
    .withColumn("processed_timestamp", F.lit(datetime.now().isoformat()))

print(f"✓ Clinical diversity: {gold_clinical_diversity.count()} providers")
gold_clinical_diversity.show(5, truncate=False)


GOLD 3: Clinical Diversity Index
✓ Clinical diversity: 2407 providers
+-----------+---------------+--------------------------+-----------------+----------+---------------------+------------+---------------+--------------------------+-----------------+----------------+---------------------+--------------------------+------------------+--------------------+-------------+-------------+
|provider_id|procedure_count|unique_patients_procedures|procedure_variety|drug_count|unique_patients_drugs|drug_variety|condition_count|unique_patients_conditions|condition_variety|total_encounters|total_unique_patients|processed_timestamp       |diversity_score   |specialization_index|is_specialist|is_generalist|
+-----------+---------------+--------------------------+-----------------+----------+---------------------+------------+---------------+--------------------------+-----------------+----------------+---------------------+--------------------------+------------------+--------------------+----------

In [15]:
print("\n" + "="*60)
print("GOLD 4: Outcome Performance Tiers")
print("="*60)

outcome_window = W.orderBy(F.col("mortality_rate"))

gold_outcome_tiers = silver_patient_outcomes \
    .withColumn("mortality_percentile", F.percent_rank().over(outcome_window)) \
    .withColumn("outcome_tier",
                F.when(F.col("mortality_percentile") <= 0.25, "Excellent")
                 .when(F.col("mortality_percentile") <= 0.50, "Good")
                 .when(F.col("mortality_percentile") <= 0.75, "Fair")
                 .otherwise("Needs Improvement")) \
    .withColumn("complexity_adjusted_mortality",
                F.when(F.col("condition_complexity") > 0,
                      F.col("mortality_rate") / F.col("condition_complexity")).otherwise(0)) \
    .withColumn("processed_timestamp", F.lit(datetime.now().isoformat()))

print(f"✓ Outcome tiers: {gold_outcome_tiers.count()} providers")
gold_outcome_tiers.show(5, truncate=False)


GOLD 4: Outcome Performance Tiers
✓ Outcome tiers: 2 providers


25/12/02 22:57:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 2

+-----------+--------------+-----------------+---------------------------+--------------------+--------------------+--------------------------+--------------------+-----------------+-----------------------------+
|provider_id|total_patients|deceased_patients|avg_condition_duration_days|condition_complexity|mortality_rate      |processed_timestamp       |mortality_percentile|outcome_tier     |complexity_adjusted_mortality|
+-----------+--------------+-----------------+---------------------------+--------------------+--------------------+--------------------------+--------------------+-----------------+-----------------------------+
|9069.0     |1             |0                |3.0                        |1                   |0.0                 |2025-12-02T22:57:55.282240|0.0                 |Excellent        |0.0                          |
|NULL       |999           |1                |NULL                       |0                   |0.001001001001001001|2025-12-02T22:57:55.282240|1.0  

25/12/02 22:57:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 2

In [16]:
print("\n" + "="*60)
print("GOLD 5: Network Connectivity Score")
print("="*60)

network_window = W.orderBy(F.col("network_centrality").desc())

gold_network_connectivity = silver_referral_network \
    .withColumn("network_rank", F.row_number().over(network_window)) \
    .withColumn("network_percentile", F.percent_rank().over(network_window)) \
    .withColumn("is_hub_provider", F.when(F.col("network_percentile") >= 0.90, 1).otherwise(0)) \
    .withColumn("referral_balance",
                F.abs(F.col("outbound_referral_count") - F.col("inbound_referral_count"))) \
    .withColumn("is_balanced_network", F.when(F.col("referral_balance") <= 2, 1).otherwise(0)) \
    .withColumn("processed_timestamp", F.lit(datetime.now().isoformat()))

print(f"✓ Network connectivity: {gold_network_connectivity.count()} providers")
gold_network_connectivity.show(5, truncate=False)


GOLD 5: Network Connectivity Score
✓ Network connectivity: 6 providers


25/12/02 22:57:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 2

+-----------+-----------------------+---------------------------+----------------------+--------------------------+------------------+--------------------------+------------+------------------+---------------+----------------+-------------------+
|provider_id|outbound_referral_count|total_patients_referred_out|inbound_referral_count|total_patients_referred_in|network_centrality|processed_timestamp       |network_rank|network_percentile|is_hub_provider|referral_balance|is_balanced_network|
+-----------+-----------------------+---------------------------+----------------------+--------------------------+------------------+--------------------------+------------+------------------+---------------+----------------+-------------------+
|2317.0     |0                      |0                          |1                     |1                         |1                 |2025-12-02T22:57:56.155885|1           |0.0               |0              |1               |1                  |
|2632.0     

25/12/02 22:57:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 2

In [17]:
print("\n" + "="*60)
print("GOLD 6: Quality Composite Subscores")
print("="*60)

quality_window = W.orderBy(F.col("readmission_rate"))

gold_quality_subscores = silver_quality_indicators \
    .withColumn("readmission_percentile", F.percent_rank().over(quality_window)) \
    .withColumn("readmission_score", (1 - F.col("readmission_percentile")) * 100) \
    .withColumn("episode_efficiency_score",
                F.when(F.col("avg_episode_length") > 0,
                      100.0 / F.col("avg_episode_length")).otherwise(0)) \
    .withColumn("continuity_score",
                F.when(F.col("avg_days_between_visits") > 0,
                      F.least(F.lit(100), 100.0 / F.col("avg_days_between_visits"))).otherwise(0)) \
    .withColumn("processed_timestamp", F.lit(datetime.now().isoformat()))

print(f"✓ Quality subscores: {gold_quality_subscores.count()} providers")
gold_quality_subscores.show(5, truncate=False)


GOLD 6: Quality Composite Subscores
✓ Quality subscores: 798 providers


25/12/02 22:57:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 2

+-----------+--------------+------------------+-----------------------+------------------+----------------+--------------------------+----------------------+-----------------+------------------------+----------------+
|provider_id|total_episodes|readmissions_30day|avg_days_between_visits|avg_episode_length|readmission_rate|processed_timestamp       |readmission_percentile|readmission_score|episode_efficiency_score|continuity_score|
+-----------+--------------+------------------+-----------------------+------------------+----------------+--------------------------+----------------------+-----------------+------------------------+----------------+
|394862.0   |1             |0                 |NULL                   |7.0               |0.0             |2025-12-02T22:57:56.863090|0.0                   |100.0            |14.285714285714286      |0.0             |
|49683.0    |4             |0                 |NULL                   |5.5               |0.0             |2025-12-02T22:57:56.8

In [18]:
print("\n" + "="*60)
print("GOLD 7: Coordination Effectiveness")
print("="*60)

coord_window = W.orderBy(F.col("coordination_complexity_score").desc())

gold_coordination_effectiveness = silver_care_coordination \
    .withColumn("coordination_rank", F.row_number().over(coord_window)) \
    .withColumn("coordination_percentile", F.percent_rank().over(coord_window)) \
    .withColumn("is_coordination_specialist", 
                F.when(F.col("coordination_percentile") >= 0.75, 1).otherwise(0)) \
    .withColumn("collaboration_intensity",
                F.col("avg_providers_per_patient") * F.col("patients_requiring_coordination")) \
    .withColumn("processed_timestamp", F.lit(datetime.now().isoformat()))

print(f"✓ Coordination effectiveness: {gold_coordination_effectiveness.count()} providers")
gold_coordination_effectiveness.show(5, truncate=False)


GOLD 7: Coordination Effectiveness
✓ Coordination effectiveness: 798 providers


25/12/02 22:57:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-----------+-------------------------+-------------------------+-------------------------------+-----------------------------+--------------------------+-----------------+-----------------------+--------------------------+-----------------------+
|provider_id|avg_providers_per_patient|max_providers_per_patient|patients_requiring_coordination|coordination_complexity_score|processed_timestamp       |coordination_rank|coordination_percentile|is_coordination_specialist|collaboration_intensity|
+-----------+-------------------------+-------------------------+-------------------------------+-----------------------------+--------------------------+-----------------+-----------------------+--------------------------+-----------------------+
|NaN        |1.0                      |1                        |8                              |0.08                         |2025-12-02T22:57:57.254818|1                |0.0                    |0                         |8.0                    |
|8823.0 

25/12/02 22:57:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 2

In [19]:
print("\n" + "="*60)
print("GOLD 8: Provider Efficiency Index")
print("="*60)

gold_efficiency_index = gold_provider_master \
    .join(gold_volume_metrics.select("provider_id", "volume_percentile", "procedure_ratio"), "provider_id") \
    .join(gold_outcome_tiers.select("provider_id", "mortality_percentile"), "provider_id") \
    .withColumn("efficiency_score",
                (F.col("volume_percentile") * 0.4 + 
                 (1 - F.col("mortality_percentile")) * 0.4 +
                 F.col("patient_per_encounter_ratio") * 0.2) * 100) \
    .withColumn("efficiency_grade",
                F.when(F.col("efficiency_score") >= 80, "A")
                 .when(F.col("efficiency_score") >= 60, "B")
                 .when(F.col("efficiency_score") >= 40, "C")
                 .otherwise("D")) \
    .withColumn("processed_timestamp", F.lit(datetime.now().isoformat()))

print(f"✓ Efficiency index: {gold_efficiency_index.count()} providers")
gold_efficiency_index.show(5, truncate=False)


GOLD 8: Provider Efficiency Index
✓ Efficiency index: 0 providers


25/12/02 22:57:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:57:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 2

+-----------+-------------+---+---+--------------------+----------------------+------------+-------------+-----------------+--------------+---------------------------+-------------------+---------------+--------------------------+-----------------+----------+---------------------+------------+---------------+--------------------------+-----------------+----------------+---------------------+-------------------+--------------+-----------------+---------------------------+--------------------+--------------+-------------------+------------+---------------------------+-----------------+---------------+--------------------+----------------+----------------+
|provider_id|provider_name|npi|dea|specialty_concept_id|specialty_source_value|care_site_id|year_of_birth|gender_concept_id|care_site_name|place_of_service_concept_id|processed_timestamp|procedure_count|unique_patients_procedures|procedure_variety|drug_count|unique_patients_drugs|drug_variety|condition_count|unique_patients_conditions|c

In [20]:
print("\n" + "="*60)
print("GOLD 9: Network Influence Metrics")
print("="*60)

gold_network_influence = gold_network_connectivity \
    .withColumn("influence_score",
                (F.col("network_centrality") * 0.5 +
                 F.col("total_patients_referred_out") * 0.3 +
                 F.col("total_patients_referred_in") * 0.2)) \
    .withColumn("referral_direction",
                F.when(F.col("outbound_referral_count") > F.col("inbound_referral_count"), "Outbound")
                 .when(F.col("inbound_referral_count") > F.col("outbound_referral_count"), "Inbound")
                 .otherwise("Balanced")) \
    .withColumn("network_role",
                F.when(F.col("is_hub_provider") == 1, "Hub")
                 .when(F.col("is_balanced_network") == 1, "Connector")
                 .otherwise("Peripheral")) \
    .withColumn("processed_timestamp", F.lit(datetime.now().isoformat()))

print(f"✓ Network influence: {gold_network_influence.count()} providers")
gold_network_influence.show(5, truncate=False)


GOLD 9: Network Influence Metrics
✓ Network influence: 6 providers


25/12/02 22:58:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:00 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 2

+-----------+-----------------------+---------------------------+----------------------+--------------------------+------------------+--------------------------+------------+------------------+---------------+----------------+-------------------+---------------+------------------+------------+
|provider_id|outbound_referral_count|total_patients_referred_out|inbound_referral_count|total_patients_referred_in|network_centrality|processed_timestamp       |network_rank|network_percentile|is_hub_provider|referral_balance|is_balanced_network|influence_score|referral_direction|network_role|
+-----------+-----------------------+---------------------------+----------------------+--------------------------+------------------+--------------------------+------------+------------------+---------------+----------------+-------------------+---------------+------------------+------------+
|2317.0     |0                      |0                          |1                     |1                         |

25/12/02 22:58:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 2

In [21]:
print("\n" + "="*60)
print("GOLD 10: Quality-Adjusted Performance")
print("="*60)

gold_quality_adjusted = gold_quality_subscores \
    .withColumn("quality_composite",
                (F.col("readmission_score") * 0.4 +
                 F.col("episode_efficiency_score") * 0.3 +
                 F.col("continuity_score") * 0.3)) \
    .withColumn("quality_tier",
                F.when(F.col("quality_composite") >= 75, "Tier 1")
                 .when(F.col("quality_composite") >= 50, "Tier 2")
                 .when(F.col("quality_composite") >= 25, "Tier 3")
                 .otherwise("Tier 4")) \
    .withColumn("is_quality_leader", F.when(F.col("quality_composite") >= 75, 1).otherwise(0)) \
    .withColumn("processed_timestamp", F.lit(datetime.now().isoformat()))

print(f"✓ Quality-adjusted performance: {gold_quality_adjusted.count()} providers")
gold_quality_adjusted.show(5, truncate=False)


GOLD 10: Quality-Adjusted Performance
✓ Quality-adjusted performance: 798 providers


25/12/02 22:58:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 2

+-----------+--------------+------------------+-----------------------+------------------+----------------+--------------------------+----------------------+-----------------+------------------------+----------------+------------------+------------+-----------------+
|provider_id|total_episodes|readmissions_30day|avg_days_between_visits|avg_episode_length|readmission_rate|processed_timestamp       |readmission_percentile|readmission_score|episode_efficiency_score|continuity_score|quality_composite |quality_tier|is_quality_leader|
+-----------+--------------+------------------+-----------------------+------------------+----------------+--------------------------+----------------------+-----------------+------------------------+----------------+------------------+------------+-----------------+
|394862.0   |1             |0                 |NULL                   |7.0               |0.0             |2025-12-02T22:58:01.263427|0.0                   |100.0            |14.285714285714286   

In [22]:
print("\n" + "="*60)
print("GOLD 11: Collaboration Network Density")
print("="*60)

gold_collaboration_density = gold_coordination_effectiveness \
    .withColumn("network_density_score",
                F.col("coordination_complexity_score") * F.col("avg_providers_per_patient")) \
    .withColumn("collaboration_type",
                F.when(F.col("is_coordination_specialist") == 1, "High Collaboration")
                 .when(F.col("avg_providers_per_patient") >= 2, "Moderate Collaboration")
                 .otherwise("Low Collaboration")) \
    .withColumn("team_based_care_indicator",
                F.when(F.col("avg_providers_per_patient") >= 3, 1).otherwise(0)) \
    .withColumn("processed_timestamp", F.lit(datetime.now().isoformat()))

print(f"✓ Collaboration density: {gold_collaboration_density.count()} providers")
gold_collaboration_density.show(5, truncate=False)


GOLD 11: Collaboration Network Density
✓ Collaboration density: 798 providers


25/12/02 22:58:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-----------+-------------------------+-------------------------+-------------------------------+-----------------------------+--------------------------+-----------------+-----------------------+--------------------------+-----------------------+---------------------+------------------+-------------------------+
|provider_id|avg_providers_per_patient|max_providers_per_patient|patients_requiring_coordination|coordination_complexity_score|processed_timestamp       |coordination_rank|coordination_percentile|is_coordination_specialist|collaboration_intensity|network_density_score|collaboration_type|team_based_care_indicator|
+-----------+-------------------------+-------------------------+-------------------------------+-----------------------------+--------------------------+-----------------+-----------------------+--------------------------+-----------------------+---------------------+------------------+-------------------------+
|NaN        |1.0                      |1               

25/12/02 22:58:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 2

In [23]:
print("\n" + "="*60)
print("GOLD 12: Outcome Attribution Scores")
print("="*60)

gold_outcome_attribution = gold_outcome_tiers \
    .join(gold_volume_metrics.select("provider_id", "total_encounters", "total_unique_patients"), "provider_id") \
    .withColumn("attribution_weight",
                F.col("total_unique_patients") / (F.col("avg_condition_duration_days") + 1)) \
    .withColumn("outcome_contribution_score",
                (1 - F.col("mortality_rate")) * F.col("attribution_weight") * 100) \
    .withColumn("complexity_adjusted_attribution",
                F.col("outcome_contribution_score") / (F.col("condition_complexity") + 1)) \
    .withColumn("processed_timestamp", F.lit(datetime.now().isoformat()))

print(f"✓ Outcome attribution: {gold_outcome_attribution.count()} providers")
gold_outcome_attribution.show(5, truncate=False)


GOLD 12: Outcome Attribution Scores


✓ Outcome attribution: 1 providers


25/12/02 22:58:03 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:03 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:03 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:03 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:03 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:03 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 2

+-----------+--------------+-----------------+---------------------------+--------------------+--------------+--------------------------+--------------------+------------+-----------------------------+----------------+---------------------+------------------+--------------------------+-------------------------------+
|provider_id|total_patients|deceased_patients|avg_condition_duration_days|condition_complexity|mortality_rate|processed_timestamp       |mortality_percentile|outcome_tier|complexity_adjusted_mortality|total_encounters|total_unique_patients|attribution_weight|outcome_contribution_score|complexity_adjusted_attribution|
+-----------+--------------+-----------------+---------------------------+--------------------+--------------+--------------------------+--------------------+------------+-----------------------------+----------------+---------------------+------------------+--------------------------+-------------------------------+
|9069.0     |1             |0              

25/12/02 22:58:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [24]:
print("\n" + "="*60)
print("GOLD 13: Network Efficiency Ratio")
print("="*60)

gold_network_efficiency = gold_network_influence \
    .join(gold_quality_adjusted.select("provider_id", "quality_composite"), "provider_id") \
    .withColumn("efficiency_ratio",
                F.when(F.col("network_centrality") > 0,
                      F.col("quality_composite") / F.col("network_centrality")).otherwise(0)) \
    .withColumn("quality_per_referral",
                F.when((F.col("total_patients_referred_out") + F.col("total_patients_referred_in")) > 0,
                      F.col("quality_composite") / (F.col("total_patients_referred_out") + F.col("total_patients_referred_in")))
                 .otherwise(0)) \
    .withColumn("is_efficient_network_provider",
                F.when(F.col("efficiency_ratio") > F.lit(0).cast("double"), 1).otherwise(0)) \
    .withColumn("processed_timestamp", F.lit(datetime.now().isoformat()))

print(f"✓ Network efficiency ratio: {gold_network_efficiency.count()} providers")
gold_network_efficiency.show(5, truncate=False)


GOLD 13: Network Efficiency Ratio
✓ Network efficiency ratio: 2 providers


25/12/02 22:58:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 2

+-----------+-----------------------+---------------------------+----------------------+--------------------------+------------------+--------------------------+------------+------------------+---------------+----------------+-------------------+---------------+------------------+------------+-----------------+----------------+--------------------+-----------------------------+
|provider_id|outbound_referral_count|total_patients_referred_out|inbound_referral_count|total_patients_referred_in|network_centrality|processed_timestamp       |network_rank|network_percentile|is_hub_provider|referral_balance|is_balanced_network|influence_score|referral_direction|network_role|quality_composite|efficiency_ratio|quality_per_referral|is_efficient_network_provider|
+-----------+-----------------------+---------------------------+----------------------+--------------------------+------------------+--------------------------+------------+------------------+---------------+----------------+------------

25/12/02 22:58:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 2

In [25]:
print("\n" + "="*60)
print("GOLD 14: Comprehensive Provider Scorecard")
print("="*60)

gold_comprehensive_scorecard = gold_provider_master \
    .join(gold_efficiency_index.select("provider_id", "efficiency_score", "efficiency_grade"), "provider_id") \
    .join(gold_quality_adjusted.select("provider_id", "quality_composite", "quality_tier"), "provider_id") \
    .join(gold_network_influence.select("provider_id", "influence_score", "network_role"), "provider_id") \
    .join(gold_collaboration_density.select("provider_id", "network_density_score", "collaboration_type"), "provider_id") \
    .withColumn("overall_performance_score",
                (F.col("efficiency_score") * 0.3 +
                 F.col("quality_composite") * 0.3 +
                 F.col("influence_score") * 0.2 +
                 F.col("network_density_score") * 0.2)) \
    .withColumn("performance_category",
                F.when(F.col("overall_performance_score") >= 75, "Exceptional")
                 .when(F.col("overall_performance_score") >= 50, "Strong")
                 .when(F.col("overall_performance_score") >= 25, "Adequate")
                 .otherwise("Developing")) \
    .withColumn("processed_timestamp", F.lit(datetime.now().isoformat()))

print(f"✓ Comprehensive scorecard: {gold_comprehensive_scorecard.count()} providers")
gold_comprehensive_scorecard.show(5, truncate=False)


GOLD 14: Comprehensive Provider Scorecard


✓ Comprehensive scorecard: 0 providers


25/12/02 22:58:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 22:58:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/02 2

+-----------+-------------+---+---+--------------------+----------------------+------------+-------------+-----------------+--------------+---------------------------+-------------------+---------------+--------------------------+-----------------+----------+---------------------+------------+---------------+--------------------------+-----------------+----------------+---------------------+-------------------+--------------+-----------------+---------------------------+--------------------+--------------+-------------------+------------+---------------------------+----------------+----------------+-----------------+------------+---------------+------------+---------------------+------------------+-------------------------+--------------------+
|provider_id|provider_name|npi|dea|specialty_concept_id|specialty_source_value|care_site_id|year_of_birth|gender_concept_id|care_site_name|place_of_service_concept_id|processed_timestamp|procedure_count|unique_patients_procedures|procedure_varie

In [26]:
print("\n" + "="*60)
print("GOLD 15: Strategic Provider Segmentation")
print("="*60)

gold_strategic_segmentation = gold_comprehensive_scorecard \
    .withColumn("segment",
                F.when((F.col("efficiency_grade") == "A") & (F.col("quality_tier") == "Tier 1"), "Star Performer")
                 .when((F.col("network_role") == "Hub") & (F.col("collaboration_type") == "High Collaboration"), "Network Leader")
                 .when((F.col("efficiency_score") >= 60) & (F.col("quality_composite") >= 50), "Solid Performer")
                 .when(F.col("is_high_volume") == 1, "High Volume")
                 .otherwise("Emerging Provider")) \
    .withColumn("strategic_priority",
                F.when(F.col("segment") == "Star Performer", "Retain & Reward")
                 .when(F.col("segment") == "Network Leader", "Expand Network")
                 .when(F.col("segment") == "Solid Performer", "Maintain")
                 .when(F.col("segment") == "High Volume", "Quality Improvement")
                 .otherwise("Develop")) \
    .withColumn("processed_timestamp", F.lit(datetime.now().isoformat()))

print(f"✓ Strategic segmentation: {gold_strategic_segmentation.count()} providers")
gold_strategic_segmentation.show(5, truncate=False)


GOLD 15: Strategic Provider Segmentation


AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `is_high_volume` cannot be resolved. Did you mean one of the following? [`drug_count`, `network_role`, `care_site_name`, `drug_variety`, `npi`].;
'Project [provider_id#1998L, provider_name#1999, npi#2000L, dea#2001, specialty_concept_id#2002, specialty_source_value#2003, care_site_id#2004L, year_of_birth#2005, gender_concept_id#2006, care_site_name#2007, place_of_service_concept_id#2008L, processed_timestamp#7885, procedure_count#2009L, unique_patients_procedures#2010L, procedure_variety#2011L, drug_count#2012L, unique_patients_drugs#2013L, drug_variety#2014L, condition_count#2015L, unique_patients_conditions#2016L, condition_variety#2017L, total_encounters#2018L, total_unique_patients#2019L, processed_timestamp#7886, ... 19 more fields]
+- Project [provider_id#1998L, provider_name#1999, npi#2000L, dea#2001, specialty_concept_id#2002, specialty_source_value#2003, care_site_id#2004L, year_of_

# STEP 5: Final Metrics (4 Ultimate KPIs)

In [38]:
print("\n" + "="*60)
print("FINAL METRIC 1: Provider Quality Composite")
print("="*60)

final_quality_composite = gold_comprehensive_scorecard \
    .select(
        "provider_id",
        "provider_name",
        "specialty_source_value",
        "quality_composite",
        "quality_tier",
        "efficiency_score",
        "efficiency_grade"
    ) \
    .withColumn("integrated_quality_score",
                (F.col("quality_composite") * 0.6 + F.col("efficiency_score") * 0.4)) \
    .withColumn("quality_status",
                F.when(F.col("integrated_quality_score") >= 75, "High Quality")
                 .when(F.col("integrated_quality_score") >= 50, "Standard Quality")
                 .otherwise("Quality Opportunity")) \
    .orderBy(F.col("integrated_quality_score").desc())

print(f"\n✓ METRIC 1 COMPLETE")
print(f"Providers analyzed: {final_quality_composite.count()}")
print(f"\nTop 5 Quality Performers:")
final_quality_composite.show(5, truncate=False)


FINAL METRIC 1: Provider Quality Composite

✓ METRIC 1 COMPLETE


Providers analyzed: 0

Top 5 Quality Performers:


25/11/22 18:37:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/22 18:37:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/22 18:37:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/22 18:37:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/22 18:37:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/22 18:37:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/22 1

+-----------+-------------+----------------------+-----------------+------------+----------------+----------------+------------------------+--------------+
|provider_id|provider_name|specialty_source_value|quality_composite|quality_tier|efficiency_score|efficiency_grade|integrated_quality_score|quality_status|
+-----------+-------------+----------------------+-----------------+------------+----------------+----------------+------------------------+--------------+
+-----------+-------------+----------------------+-----------------+------------+----------------+----------------+------------------------+--------------+



In [40]:
print("\n" + "="*60)
print("FINAL METRIC 2: Network Efficiency Score")
print("="*60)

final_network_efficiency = gold_network_efficiency \
    .join(
        gold_network_influence.select(
            F.col("provider_id").alias("prov_id"),
            F.col("network_role").alias("net_role"),
            F.col("referral_direction").alias("ref_dir")
        ), 
        gold_network_efficiency.provider_id == F.col("prov_id"),
        "left"
    ) \
    .select(
        gold_network_efficiency.provider_id,
        F.col("net_role").alias("network_role"),
        F.col("ref_dir").alias("referral_direction"),
        gold_network_efficiency.efficiency_ratio,
        gold_network_efficiency.quality_per_referral,
        gold_network_efficiency.is_efficient_network_provider
    ) \
    .withColumn("network_efficiency_score",
                (F.col("efficiency_ratio") * 50 + F.col("quality_per_referral") * 50)) \
    .withColumn("efficiency_rating",
                F.when(F.col("network_efficiency_score") >= 75, "Highly Efficient")
                 .when(F.col("network_efficiency_score") >= 50, "Efficient")
                 .when(F.col("network_efficiency_score") >= 25, "Moderately Efficient")
                 .otherwise("Inefficient")) \
    .orderBy(F.col("network_efficiency_score").desc())

print(f"\n✓ METRIC 2 COMPLETE")
print(f"Providers in network: {final_network_efficiency.count()}")
print(f"\nTop 5 Network Efficient Providers:")
final_network_efficiency.show(5, truncate=False)


FINAL METRIC 2: Network Efficiency Score

✓ METRIC 2 COMPLETE
Providers in network: 2

Top 5 Network Efficient Providers:


25/11/22 18:38:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/22 18:38:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/22 18:38:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/22 18:38:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/22 18:38:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/22 18:38:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/22 1

+-----------+------------+------------------+----------------+--------------------+-----------------------------+------------------------+-----------------+
|provider_id|network_role|referral_direction|efficiency_ratio|quality_per_referral|is_efficient_network_provider|network_efficiency_score|efficiency_rating|
+-----------+------------+------------------+----------------+--------------------+-----------------------------+------------------------+-----------------+
|2632.0     |Connector   |Outbound          |50.0            |50.0                |1                            |5000.0                  |Highly Efficient |
|61665.0    |Connector   |Outbound          |46.0            |46.0                |1                            |4600.0                  |Highly Efficient |
+-----------+------------+------------------+----------------+--------------------+-----------------------------+------------------------+-----------------+



25/11/22 18:38:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/22 18:38:13 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [42]:
print("\n" + "="*60)
print("FINAL METRIC 3: Patient Outcome Attribution")
print("="*60)

final_outcome_attribution = gold_outcome_attribution \
    .select(
        "provider_id",
        "outcome_tier",
        "outcome_contribution_score",
        "complexity_adjusted_attribution",
        "attribution_weight",
        "total_patients",
        "condition_complexity"
    ) \
    .withColumn("patient_outcome_impact",
                F.col("outcome_contribution_score") * F.col("complexity_adjusted_attribution")) \
    .withColumn("attribution_category",
                F.when(F.col("patient_outcome_impact") >= 1000, "High Impact")
                 .when(F.col("patient_outcome_impact") >= 500, "Moderate Impact")
                 .when(F.col("patient_outcome_impact") >= 100, "Standard Impact")
                 .otherwise("Emerging Impact")) \
    .orderBy(F.col("patient_outcome_impact").desc())

print(f"\n✓ METRIC 3 COMPLETE")
print(f"Providers with outcome data: {final_outcome_attribution.count()}")
print(f"\nTop 5 Outcome Contributors:")
final_outcome_attribution.show(5, truncate=False)


FINAL METRIC 3: Patient Outcome Attribution

✓ METRIC 3 COMPLETE
Providers with outcome data: 1

Top 5 Outcome Contributors:


25/11/22 18:39:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/22 18:39:04 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/22 18:39:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/22 18:39:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/22 18:39:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/22 18:39:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/22 1

+-----------+------------+--------------------------+-------------------------------+------------------+--------------+--------------------+----------------------+--------------------+
|provider_id|outcome_tier|outcome_contribution_score|complexity_adjusted_attribution|attribution_weight|total_patients|condition_complexity|patient_outcome_impact|attribution_category|
+-----------+------------+--------------------------+-------------------------------+------------------+--------------+--------------------+----------------------+--------------------+
|9069.0     |Excellent   |25.0                      |12.5                           |0.25              |1             |1                   |312.5                 |Standard Impact     |
+-----------+------------+--------------------------+-------------------------------+------------------+--------------+--------------------+----------------------+--------------------+



25/11/22 18:39:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/22 18:39:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/22 18:39:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/22 18:39:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/22 18:39:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/22 18:39:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/22 1

In [44]:
print("\n" + "="*60)
print("FINAL METRIC 4: Care Coordination Index")
print("="*60)

final_care_coordination = gold_collaboration_density \
    .select(
        "provider_id",
        "collaboration_type",
        "network_density_score",
        "avg_providers_per_patient",
        "patients_requiring_coordination",
        "team_based_care_indicator",
        "collaboration_intensity"
    ) \
    .withColumn("coordination_index",
                (F.col("network_density_score") * 0.4 +
                 F.col("collaboration_intensity") * 0.3 +
                 F.col("avg_providers_per_patient") * 10 * 0.3)) \
    .withColumn("coordination_excellence",
                F.when(F.col("coordination_index") >= 50, "Excellent Coordination")
                 .when(F.col("coordination_index") >= 30, "Good Coordination")
                 .when(F.col("coordination_index") >= 15, "Adequate Coordination")
                 .otherwise("Limited Coordination")) \
    .orderBy(F.col("coordination_index").desc())

print(f"\n✓ METRIC 4 COMPLETE")
print(f"Providers with coordination data: {final_care_coordination.count()}")
print(f"\nTop 5 Care Coordinators:")
final_care_coordination.show(5, truncate=False)


FINAL METRIC 4: Care Coordination Index

✓ METRIC 4 COMPLETE
Providers with coordination data: 798

Top 5 Care Coordinators:


25/11/22 18:41:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/22 18:41:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/22 18:41:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/22 18:41:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/22 18:41:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/22 18:41:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/11/22 1

+-----------+----------------------+---------------------+-------------------------+-------------------------------+-------------------------+-----------------------+------------------+-----------------------+
|provider_id|collaboration_type    |network_density_score|avg_providers_per_patient|patients_requiring_coordination|team_based_care_indicator|collaboration_intensity|coordination_index|coordination_excellence|
+-----------+----------------------+---------------------+-------------------------+-------------------------------+-------------------------+-----------------------+------------------+-----------------------+
|329084.0   |Moderate Collaboration|0.04                 |2.0                      |1                              |0                        |2.0                    |6.616             |Limited Coordination   |
|81918.0    |Moderate Collaboration|0.04                 |2.0                      |1                              |0                        |2.0               

# STEP 6: Build DAG (Bronze → Silver 6 → Gold 15 → Metrics 4)

In [27]:
# ============================================================================
# STEP 7: AUTO-PARSING EDGE-CENTRIC LINEAGE DAG (v3) - Provider Performance
# ============================================================================
# Features:
# - withColumn: extracts ALL columns (including chained operations)
# - agg metrics: extracts F.sum, F.count, F.avg, F.countDistinct, etc.
# - join: detects join operations with on/how parameters
# - filter: detects filter operations with conditions
# - union: detects union operations between DataFrames
# - withColumnRenamed: detects column rename operations
# - Window functions: detects F.lead, percent_rank, etc.
# - Handles intermediate DataFrames (referral_network, outbound_referrals, etc.)
# - Supports 4-layer architecture: Bronze → Silver → Gold → Metric
# ============================================================================

print("\n" + "="*80)
print("STEP 7: AUTO-PARSING EDGE-CENTRIC LINEAGE DAG (v3)")
print("="*80)

import json
import re
import networkx as nx
from datetime import datetime
from typing import List, Dict, Tuple

# ============================================================================
# CONFIGURATION - Update these paths as needed
# ============================================================================
NOTEBOOK_FILE = "./4_provider_performance_network.ipynb"

# Use existing LOCAL_DATA_DIR or set default
try:
    LOCAL_DATA_DIR
except NameError:
    LOCAL_DATA_DIR = "./4_data"
    import os
    os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

# ============================================================================
# NODE DESCRIPTIONS - 4번 노트북 테이블 설명
# ============================================================================
NODE_DESCRIPTIONS = {
    # Bronze Layer (10 tables)
    "bronze_provider": "OMOP: Healthcare provider information",
    "bronze_care_site": "OMOP: Healthcare facility information",
    "bronze_person": "OMOP: Patient demographics",
    "bronze_death": "OMOP: Patient death records",
    "bronze_procedure": "OMOP: Procedure occurrence records",
    "bronze_drug": "OMOP: Drug exposure records",
    "bronze_condition": "OMOP: Condition occurrence records",
    "bronze_observation": "OMOP: Clinical observations",
    "bronze_medicare": "Medicare: Physicians and other supplier 2014",
    "bronze_taxonomy": "NPPES: Healthcare provider taxonomy codes",
    
    # Intermediate DataFrames
    "procedure_referrals": "Intermediate: Procedure-based referral source",
    "drug_providers": "Intermediate: Drug exposure provider targets",
    "referral_network": "Intermediate: Provider-to-provider referral network",
    "outbound_referrals": "Intermediate: Outbound referral aggregation",
    "inbound_referrals": "Intermediate: Inbound referral aggregation",
    "patient_provider_count": "Intermediate: Patient-provider relationship counts",
    "coordination_metrics": "Intermediate: Care coordination metrics",
    "readmission_proxy": "Intermediate: Readmission calculation proxy",
    "quality_indicators": "Intermediate: Quality indicator calculations",
    "patient_outcomes": "Intermediate: Patient outcomes aggregation",
    
    # Silver Layer (6 tables)
    "silver_provider_demographics": "Provider demographics with care site enrichment",
    "silver_clinical_activity": "Provider clinical activity metrics (procedures, drugs, conditions)",
    "silver_patient_outcomes": "Patient outcomes by provider (mortality, complexity)",
    "silver_referral_network": "Provider referral network metrics (in/outbound)",
    "silver_quality_indicators": "Provider quality indicators (readmission, episode length)",
    "silver_care_coordination": "Care coordination complexity metrics",
    
    # Gold Layer (15 tables)
    "gold_provider_master": "Provider master profile integrating all silver data",
    "gold_activity_metrics": "Normalized activity metrics and efficiency scores",
    "gold_outcome_scores": "Outcome scores with mortality benchmarks",
    "gold_network_analysis": "Network centrality and influence analysis",
    "gold_network_influence": "Network influence and referral direction",
    "gold_quality_subscores": "Quality composite subscores",
    "gold_coordination_effectiveness": "Coordination effectiveness metrics",
    "gold_network_efficiency": "Network efficiency ratios",
    "gold_referral_patterns": "Referral pattern analysis",
    "gold_quality_adjusted": "Quality-adjusted performance scores",
    "gold_collaboration_density": "Collaboration network density metrics",
    "gold_outcome_attribution": "Outcome attribution by provider",
    "gold_performance_tiers": "Provider performance tier classification",
    "gold_efficiency_ranking": "Efficiency ranking across providers",
    "gold_final_integration": "Final integration of all gold metrics",
    
    # Metric Layer (4 final KPIs)
    "final_quality_composite": "FINAL: Provider Quality Composite Score",
    "final_network_efficiency": "FINAL: Network Efficiency Score",
    "final_outcome_attribution": "FINAL: Patient Outcome Attribution",
    "final_care_coordination": "FINAL: Care Coordination Index",
}

# ============================================================================
# LINEAGE PARSER CLASS
# ============================================================================
class LineageParserV3:
    """
    Improved PySpark Lineage Parser for Provider Performance Pipeline
    - Parses notebook code cells to extract DataFrame transformations
    - Supports: withColumn, groupBy+agg, join, filter, select, fillna, union, withColumnRenamed
    - Handles intermediate DataFrames and complex referral network logic
    - Outputs edge-centric lineage format for RAG retrieval
    """
    
    def __init__(self):
        self.edges = []
        self.G = nx.DiGraph()
    
    def get_layer(self, node_id: str) -> str:
        """Extract layer from node ID (bronze/silver/gold/metric)"""
        node_lower = node_id.lower()
        if 'bronze' in node_lower:
            return 'bronze'
        elif 'silver' in node_lower:
            return 'silver'
        elif 'gold' in node_lower:
            return 'gold'
        elif 'final' in node_lower or 'metric' in node_lower:
            return 'metric'
        return 'intermediate'
    
    def extract_all_withcolumns(self, code_block: str) -> List[str]:
        """Extract all column names from withColumn operations"""
        columns = []
        normalized = re.sub(r'\s+', ' ', code_block)
        pattern = r'\.withColumn\s*\(\s*["\']([^"\']+)["\']'
        for match in re.finditer(pattern, normalized):
            col = match.group(1)
            if col not in columns:
                columns.append(col)
        return columns
    
    def extract_column_renames(self, code_block: str) -> List[Dict[str, str]]:
        """Extract column renames from withColumnRenamed operations"""
        renames = []
        normalized = re.sub(r'\s+', ' ', code_block)
        pattern = r'\.withColumnRenamed\s*\(\s*["\']([^"\']+)["\']\s*,\s*["\']([^"\']+)["\']\s*\)'
        for match in re.finditer(pattern, normalized):
            renames.append({"from": match.group(1), "to": match.group(2)})
        return renames
    
    def extract_agg_metrics(self, agg_block: str) -> Dict[str, str]:
        """Extract all metrics from agg block (F.sum, F.count, etc.)"""
        metrics = {}
        normalized = re.sub(r'\s+', ' ', agg_block)
        
        patterns = [
            r'F\s*\.\s*(\w+)\s*\(\s*"([^"]*)"\s*\)\s*\.\s*alias\s*\(\s*"([^"]+)"\s*\)',
            r"F\s*\.\s*(\w+)\s*\(\s*'([^']*)'\s*\)\s*\.\s*alias\s*\(\s*'([^']+)'\s*\)",
            r'F\s*\.\s*(\w+)\s*\(\s*"\*"\s*\)\s*\.\s*alias\s*\(\s*"([^"]+)"\s*\)',
        ]
        
        for pattern in patterns[:2]:
            for match in re.finditer(pattern, normalized):
                func = match.group(1)
                col = match.group(2)
                alias = match.group(3)
                metrics[alias] = f"{func}({col})"
        
        for match in re.finditer(patterns[2], normalized):
            func = match.group(1)
            alias = match.group(2)
            metrics[alias] = f"{func}(*)"
        
        return metrics
    
    def extract_groupby_cols(self, groupby_str: str) -> List[str]:
        """Extract column names from groupBy clause"""
        cols = []
        for match in re.finditer(r'["\']([^"\']+)["\']', groupby_str):
            cols.append(match.group(1))
        return cols
    
    def extract_filter_condition(self, code_block: str) -> str:
        """Extract filter condition"""
        pattern = r'\.filter\s*\(\s*([^)]+)\s*\)'
        match = re.search(pattern, code_block)
        if match:
            cond = match.group(1).strip()
            cond = re.sub(r'F\.col\s*\(\s*["\']([^"\']+)["\']\s*\)', r'\1', cond)
            return cond[:100]
        return None
    
    def extract_join_info(self, code_block: str) -> List[Dict[str, str]]:
        """Extract ALL join information"""
        joins = []
        normalized = re.sub(r'\s+', ' ', code_block)
        
        # Pattern 1: .join(df, condition, "type")
        pattern_complex = r'\.join\s*\(\s*(\w+)(?:\.alias\s*\(\s*["\'](\w+)["\']\s*\))?\s*,\s*([^,]+)\s*,\s*["\'](\w+)["\']\s*\)'
        
        for match in re.finditer(pattern_complex, normalized):
            other = match.group(1)
            alias = match.group(2) if match.group(2) else None
            on_clause = match.group(3).strip()
            how = match.group(4)
            
            col_match = re.search(r'["\']([^"\']+)["\']', on_clause)
            on_col = col_match.group(1) if col_match else on_clause[:50]
            
            joins.append({
                "other": alias if alias else other,
                "on": on_col,
                "how": how
            })
        
        # Pattern 2: .join(df, df.col == df2.col, "type")
        pattern_eq = r'\.join\s*\(\s*(\w+)\s*,\s*(\w+\.\w+)\s*==\s*(\w+\.\w+)\s*,\s*["\'](\w+)["\']\s*\)'
        
        if not joins:
            for match in re.finditer(pattern_eq, normalized):
                other = match.group(1)
                on_col = f"{match.group(2)} == {match.group(3)}"
                how = match.group(4)
                joins.append({"other": other, "on": on_col, "how": how})
        
        # Pattern 3: Simple .join(df, "col", "how")
        pattern_simple = r'\.join\s*\(\s*(\w+)\s*,\s*["\']([^"\']+)["\']\s*,\s*["\'](\w+)["\']\s*\)'
        
        if not joins:
            for match in re.finditer(pattern_simple, normalized):
                joins.append({
                    "other": match.group(1),
                    "on": match.group(2),
                    "how": match.group(3)
                })
        
        # Pattern 4: .join(df, "col") - no join type specified (default inner)
        pattern_default = r'\.join\s*\(\s*(\w+)\s*,\s*["\']([^"\']+)["\']\s*\)'
        
        if not joins:
            for match in re.finditer(pattern_default, normalized):
                joins.append({
                    "other": match.group(1),
                    "on": match.group(2),
                    "how": "inner"
                })
        
        return joins if joins else None
    
    def extract_union_info(self, code_block: str) -> List[str]:
        """Extract union operations"""
        unions = []
        normalized = re.sub(r'\s+', ' ', code_block)
        pattern = r'\.union\s*\(\s*(\w+)(?:\.[^)]+)?\s*\)'
        for match in re.finditer(pattern, normalized):
            unions.append(match.group(1))
        return unions
    
    def extract_select_cols(self, code_block: str) -> List[str]:
        """Extract columns from select operation"""
        cols = []
        pattern = r'\.select\s*\(\s*([^)]+)\s*\)'
        match = re.search(pattern, code_block)
        if match:
            select_str = match.group(1)
            for col_match in re.finditer(r'["\']([^"\']+)["\']', select_str):
                cols.append(col_match.group(1))
        return cols
    
    def extract_fillna_cols(self, code_block: str) -> bool:
        """Check if fillna is used"""
        return '.fillna(' in code_block
    
    def extract_window_functions(self, code_block: str) -> List[str]:
        """Extract Window function usage"""
        funcs = []
        if 'F.lead(' in code_block:
            funcs.append('lead')
        if 'F.lag(' in code_block:
            funcs.append('lag')
        if 'percent_rank()' in code_block:
            funcs.append('percent_rank')
        if 'row_number()' in code_block:
            funcs.append('row_number')
        if 'rank()' in code_block and 'percent_rank' not in code_block:
            funcs.append('rank')
        return funcs
    
    def parse_assignment(self, code: str) -> List[Dict]:
        """Parse DataFrame assignment statements from code"""
        edges = []
        lines = code.split('\n')
        
        i = 0
        while i < len(lines):
            line = lines[i].strip()
            
            assign_match = re.match(r'(\w+)\s*=\s*\(?\s*(\w+)?', line)
            if assign_match and '=' in line and not line.startswith('#'):
                target = assign_match.group(1)
                
                # Skip non-DataFrame variables
                skip = {'spark', 'print', 'if', 'for', 'while', 'def', 'class', 
                       'client', 'query', 'output_file', 'stats', 'rag_data', 
                       'dag_file', 'G', 'lineage_edges', 'edges', 'node_attrs',
                       'label', 'layer', 'in_degree', 'out_degree', 'parents',
                       'children', 'texts', 'quality_window', 'coordination_window',
                       'network_window', 'activity_window'}
                if target.lower() in skip or target in skip:
                    i += 1
                    continue
                
                # Collect full statement
                full_statement = line
                paren_count = line.count('(') - line.count(')')
                backslash_continue = line.rstrip().endswith('\\')
                j = i + 1
                
                while (paren_count > 0 or backslash_continue) and j < len(lines):
                    next_line = lines[j]
                    full_statement += '\n' + next_line
                    paren_count += next_line.count('(') - next_line.count(')')
                    backslash_continue = next_line.rstrip().endswith('\\')
                    j += 1
                
                # Process relevant DataFrames
                keywords = ['bronze', 'silver', 'gold', 'final', 'metric',
                           'referral', 'provider', 'patient', 'coordination',
                           'quality', 'outcome', 'activity', 'network']
                if any(x in full_statement.lower() for x in keywords):
                    edge = self.parse_single_assignment(target, full_statement)
                    if edge:
                        edges.append(edge)
                
                i = j
            else:
                i += 1
        
        return edges
    
    def parse_single_assignment(self, target: str, full_statement: str) -> Dict:
        """Parse a single DataFrame assignment statement"""
        # Find source DataFrame
        source_match = re.search(r'=\s*\(?\s*(\w+)(?:\s*\\)?\s*\.', full_statement)
        if not source_match:
            return None
        
        source = source_match.group(1)
        
        # Skip non-DataFrame sources
        if source.lower() in {'spark', 'f', 'w', 'os', 'pd', 'nx', 'json', 'print', 'datetime'}:
            return None
        
        sources = [source]
        op_types = []
        
        # Parse unions (important for provider performance pipeline)
        union_sources = self.extract_union_info(full_statement)
        if union_sources:
            for u in union_sources:
                if u not in sources:
                    sources.append(u)
            op_types.append('union')
        
        # Parse joins
        joins_info = self.extract_join_info(full_statement)
        if joins_info:
            for join in joins_info:
                if join['other'] not in sources:
                    sources.append(join['other'])
            op_types.append('join')
        
        # Parse filter
        filter_cond = self.extract_filter_condition(full_statement)
        if filter_cond:
            op_types.append('filter')
        
        # Parse groupBy
        groupby_match = re.search(r'\.groupBy\s*\(\s*([^)]+)\s*\)', full_statement)
        groupby_cols = []
        if groupby_match:
            groupby_cols = self.extract_groupby_cols(groupby_match.group(1))
            op_types.append('agg')
        
        # Parse agg metrics
        metrics = {}
        agg_start = full_statement.find('.agg(')
        if agg_start != -1:
            paren_count = 0
            content_start = agg_start + 5
            content_end = content_start
            
            for idx, char in enumerate(full_statement[agg_start:]):
                if char == '(':
                    paren_count += 1
                elif char == ')':
                    paren_count -= 1
                    if paren_count == 0:
                        content_end = agg_start + idx
                        break
            
            agg_content = full_statement[content_start:content_end]
            metrics = self.extract_agg_metrics(agg_content)
        
        # Parse withColumn
        columns = self.extract_all_withcolumns(full_statement)
        if columns:
            op_types.append('withColumn')
        
        # Parse withColumnRenamed
        renames = self.extract_column_renames(full_statement)
        if renames:
            op_types.append('rename')
        
        # Parse select
        select_cols = self.extract_select_cols(full_statement)
        if select_cols and not op_types:
            op_types.append('select')
        
        # Parse fillna
        if self.extract_fillna_cols(full_statement):
            op_types.append('fillna')
        
        # Parse window functions
        window_funcs = self.extract_window_functions(full_statement)
        if window_funcs:
            op_types.append('window')
        
        # Skip if no operations found
        if not op_types:
            return None
        
        op_str = '+'.join(op_types)
        
        # Build operation dict
        operation = {"op": op_str}
        if joins_info:
            operation["joins"] = joins_info
        if union_sources:
            operation["unions"] = union_sources
        if filter_cond:
            operation["filter"] = filter_cond
        if groupby_cols:
            operation["groupBy"] = groupby_cols
        if metrics:
            operation["metrics"] = metrics
        if columns:
            operation["columns"] = columns
        if renames:
            operation["renames"] = renames
        if select_cols:
            operation["select"] = select_cols
        if window_funcs:
            operation["window_functions"] = window_funcs
        
        # Generate description text
        text_parts = [f"{target} is created from {source}:"]
        if union_sources:
            text_parts.append(f"unions with {union_sources}")
        if joins_info:
            for j in joins_info:
                text_parts.append(f"{j['how']}-joins with {j['other']} on {j['on']}")
        if filter_cond:
            text_parts.append(f"filters by {filter_cond[:50]}")
        if groupby_cols:
            text_parts.append(f"groups by {groupby_cols}")
        if metrics:
            text_parts.append(f"computes {list(metrics.keys())}")
        if columns:
            text_parts.append(f"adds columns {columns}")
        if renames:
            text_parts.append(f"renames {[r['from'] + '->' + r['to'] for r in renames]}")
        if window_funcs:
            text_parts.append(f"uses window functions {window_funcs}")
        
        text = " ".join(text_parts) + "."
        
        return {
            "id": f"{' + '.join(sources)} -> {target} ({op_str})",
            "source_nodes": sources,
            "target_node": target,
            "operation": operation,
            "text": text
        }
    
    def parse_notebook(self, notebook_path: str) -> Tuple[List[Dict], nx.DiGraph]:
        """Parse entire notebook file and extract lineage"""
        print(f"\n📖 Reading notebook: {notebook_path}")
        
        with open(notebook_path, 'r', encoding='utf-8') as f:
            notebook = json.load(f)
        
        code_cells = []
        for cell in notebook['cells']:
            if cell['cell_type'] == 'code':
                source = cell.get('source', [])
                code = ''.join(source) if isinstance(source, list) else source
                code_cells.append(code)
        
        print(f"   Found {len(code_cells)} code cells")
        
        all_edges = []
        for code in code_cells:
            edges = self.parse_assignment(code)
            all_edges.extend(edges)
        
        # Deduplicate by target
        seen_targets = {}
        for edge in all_edges:
            target = edge['target_node']
            if target not in seen_targets:
                seen_targets[target] = edge
            else:
                existing = seen_targets[target]
                existing_score = len(existing.get('operation', {}).get('columns', [])) + \
                                len(existing.get('operation', {}).get('metrics', {})) + \
                                len(existing['source_nodes'])
                new_score = len(edge.get('operation', {}).get('columns', [])) + \
                           len(edge.get('operation', {}).get('metrics', {})) + \
                           len(edge['source_nodes'])
                if new_score > existing_score:
                    seen_targets[target] = edge
        
        self.edges = list(seen_targets.values())
        self.build_graph()
        
        print(f"   Extracted {len(self.edges)} transformation edges")
        
        return self.edges, self.G
    
    def build_graph(self):
        """Build NetworkX DAG from edges"""
        self.G = nx.DiGraph()
        
        for edge in self.edges:
            target = edge['target_node']
            if not self.G.has_node(target):
                self.G.add_node(target, id=target, label=target, layer=self.get_layer(target))
            
            for src in edge['source_nodes']:
                if not self.G.has_node(src):
                    self.G.add_node(src, id=src, label=src, layer=self.get_layer(src))
                self.G.add_edge(src, target, etype="consume")
    
    def generate_rag_data(self) -> List[Dict]:
        """Generate RAG-compatible node data"""
        rag_data = []
        
        for node_id in self.G.nodes():
            layer = self.get_layer(node_id)
            in_deg = self.G.in_degree(node_id)
            out_deg = self.G.out_degree(node_id)
            parents = list(self.G.predecessors(node_id))
            children = list(self.G.successors(node_id))
            
            description = NODE_DESCRIPTIONS.get(node_id, node_id)
            
            texts = [
                description,
                f"Layer: {layer}",
                f"Incoming edges: {in_deg}, Outgoing edges: {out_deg}"
            ]
            if parents:
                texts.append(f"Consumes: {', '.join(parents)}")
            if children:
                texts.append(f"Feeds into: {', '.join(children)}")
            
            rag_data.append({"id": node_id, "texts": texts})
        
        return rag_data


# ============================================================================
# MAIN EXECUTION
# ============================================================================
print("\n" + "="*60)
print("Parsing Notebook for PySpark Transformations")
print("="*60)

parser = LineageParserV3()

try:
    edges, G = parser.parse_notebook(NOTEBOOK_FILE)
    rag_data = parser.generate_rag_data()
    
    # Statistics
    bronze_count = len([n for n in G.nodes() if parser.get_layer(n) == 'bronze'])
    silver_count = len([n for n in G.nodes() if parser.get_layer(n) == 'silver'])
    gold_count = len([n for n in G.nodes() if parser.get_layer(n) == 'gold'])
    metric_count = len([n for n in G.nodes() if parser.get_layer(n) == 'metric'])
    intermediate_count = len([n for n in G.nodes() if parser.get_layer(n) == 'intermediate'])
    
    print("\n" + "="*60)
    print("Parsing Results")
    print("="*60)
    print(f"  Total nodes: {G.number_of_nodes()}")
    print(f"  Total graph edges: {G.number_of_edges()}")
    print(f"  Lineage records: {len(edges)}")
    print(f"  Bronze: {bronze_count}, Silver: {silver_count}, Gold: {gold_count}, Metric: {metric_count}")
    print(f"  Intermediate: {intermediate_count}")
    print(f"  Is DAG: {nx.is_directed_acyclic_graph(G)}")
    
    # ========================================================================
    # SAVE OUTPUTS
    # ========================================================================
    print("\n" + "="*60)
    print("Saving Outputs")
    print("="*60)
    
    lineage_file = f"{LOCAL_DATA_DIR}/provider_performance_lineage_edges_auto.json"
    with open(lineage_file, 'w', encoding='utf-8') as f:
        json.dump(edges, f, indent=2, ensure_ascii=False)
    print(f"✔ Saved lineage: {lineage_file}")
    
    rag_file = f"{LOCAL_DATA_DIR}/provider_performance_rag_data_auto.json"
    with open(rag_file, 'w', encoding='utf-8') as f:
        json.dump(rag_data, f, indent=2, ensure_ascii=False)
    print(f"✔ Saved RAG data: {rag_file}")
    
    dag_file = f"{LOCAL_DATA_DIR}/provider_performance_dag_auto.graphml"
    nx.write_graphml(G, dag_file)
    print(f"✔ Saved DAG: {dag_file}")
    
    stats = {
        "pipeline": "Provider Performance Network (Auto-Parsed v3)",
        "total_nodes": G.number_of_nodes(),
        "total_edges": G.number_of_edges(),
        "lineage_edges": len(edges),
        "bronze_nodes": bronze_count,
        "silver_nodes": silver_count,
        "gold_nodes": gold_count,
        "metric_nodes": metric_count,
        "intermediate_nodes": intermediate_count,
        "is_dag": nx.is_directed_acyclic_graph(G),
        "timestamp": datetime.now().isoformat(),
        "source_notebook": NOTEBOOK_FILE
    }
    
    stats_file = f"{LOCAL_DATA_DIR}/dag_statistics_auto.json"
    with open(stats_file, 'w', encoding='utf-8') as f:
        json.dump(stats, f, indent=2, ensure_ascii=False)
    print(f"✔ Saved statistics: {stats_file}")
    
    # ========================================================================
    # DISPLAY RESULTS
    # ========================================================================
    print("\n" + "="*80)
    print("AUTO-PARSING COMPLETE (v3)")
    print("="*80)
    
    print("\n--- Extracted Transformations ---\n")
    for i, edge in enumerate(edges, 1):
        print(f"{i}. [{edge['operation']['op']}] {edge['id']}")
        print(f"   Sources: {edge['source_nodes']}")
        print(f"   Target:  {edge['target_node']}")
        
        op = edge['operation']
        if 'columns' in op and op['columns']:
            print(f"   Columns: {op['columns']}")
        if 'groupBy' in op and op['groupBy']:
            print(f"   GroupBy: {op['groupBy']}")
        if 'metrics' in op and op['metrics']:
            print(f"   Metrics: {op['metrics']}")
        if 'joins' in op:
            for j in op['joins']:
                print(f"   Join:    {j['how']} on '{j['on']}' with {j['other']}")
        if 'unions' in op:
            print(f"   Union:   {op['unions']}")
        if 'filter' in op:
            print(f"   Filter:  {op['filter']}")
        if 'window_functions' in op:
            print(f"   Window:  {op['window_functions']}")
        if 'renames' in op:
            print(f"   Renames: {op['renames']}")
        print(f"   Text:    {edge['text']}")
        print()
    
    print("="*80)
    print(f"Total: {len(edges)} transformations, {G.number_of_nodes()} nodes")
    print(f"Files saved to: {LOCAL_DATA_DIR}/")
    print("="*80)

except FileNotFoundError:
    print(f"\n❌ Error: Notebook file not found: {NOTEBOOK_FILE}")
    print("   Update NOTEBOOK_FILE path at the top of this cell.")
except Exception as e:
    print(f"\n❌ Error: {e}")
    import traceback
    traceback.print_exc()


STEP 7: AUTO-PARSING EDGE-CENTRIC LINEAGE DAG (v3)

Parsing Notebook for PySpark Transformations

📖 Reading notebook: ./4_provider_performance_network.ipynb
   Found 38 code cells
   Extracted 35 transformation edges

Parsing Results
  Total nodes: 45
  Total graph edges: 46
  Lineage records: 35
  Bronze: 7, Silver: 6, Gold: 15, Metric: 5
  Intermediate: 12
  Is DAG: True

Saving Outputs
✔ Saved lineage: ./4_data/provider_performance_lineage_edges_auto.json
✔ Saved RAG data: ./4_data/provider_performance_rag_data_auto.json
✔ Saved DAG: ./4_data/provider_performance_dag_auto.graphml
✔ Saved statistics: ./4_data/dag_statistics_auto.json

AUTO-PARSING COMPLETE (v3)

--- Extracted Transformations ---

1. [join] bronze_provider + bronze_care_site -> silver_provider_demographics (join)
   Sources: ['bronze_provider', 'bronze_care_site']
   Target:  silver_provider_demographics
   Join:    left on 'bronze_provider.care_site_id == bronze_care_site.c' with bronze_care_site
   Text:    silver_

# STEP 7: Summary

In [48]:
print("\n" + "="*80)
print("PROVIDER PERFORMANCE & NETWORK ANALYTICS COMPLETE")
print("="*80)

print(f"\nData Sources:")
print(f"  OMOP Clinical (8 tables)")
print(f"  Medicare Provider (1 table)")
print(f"  NPPES Taxonomy (1 table)")

print(f"\nPipeline Architecture:")
print(f"  Bronze Layer: {len(bronze_nodes)} tables (raw data)")
print(f"  Silver Layer: {len(silver_nodes)} tables (provider profiling)")
print(f"  Gold Layer: {len(gold_nodes)} tables (vertical analytics)")
print(f"  Final Metrics: {len(metric_nodes)} KPIs")

print(f"\nFinal Metrics:")
print(f"  1. Provider Quality Composite: {final_quality_composite.count()} providers")
print(f"  2. Network Efficiency Score: {final_network_efficiency.count()} providers")
print(f"  3. Patient Outcome Attribution: {final_outcome_attribution.count()} providers")
print(f"  4. Care Coordination Index: {final_care_coordination.count()} providers")

print(f"\nDAG:")
print(f"  Total nodes: {G.number_of_nodes()}")
print(f"  Total edges: {G.number_of_edges()}")
print(f"  Format: GraphML + RAG JSON")

print(f"\nCost Optimization:")
print(f"  Strategy: LIMIT {LIMIT} on all BigQuery reads")
print(f"  Storage: Local CSV (Pandas → Spark)")
print(f"  Total tables processed: {len(bronze_nodes)}")

print("\n" + "="*80)
print("✓ Ready for Marquez lineage tracking")
print("✓ Ready for RAG pipeline integration")
print("="*80)


PROVIDER PERFORMANCE & NETWORK ANALYTICS COMPLETE

Data Sources:
  OMOP Clinical (8 tables)
  Medicare Provider (1 table)
  NPPES Taxonomy (1 table)

Pipeline Architecture:
  Bronze Layer: 10 tables (raw data)
  Silver Layer: 6 tables (provider profiling)
  Gold Layer: 15 tables (vertical analytics)
  Final Metrics: 4 KPIs

Final Metrics:


  1. Provider Quality Composite: 0 providers
  2. Network Efficiency Score: 2 providers
  3. Patient Outcome Attribution: 1 providers
  4. Care Coordination Index: 798 providers

DAG:
  Total nodes: 35
  Total edges: 47
  Format: GraphML + RAG JSON

Cost Optimization:
  Strategy: LIMIT 1000 on all BigQuery reads
  Storage: Local CSV (Pandas → Spark)
  Total tables processed: 10

✓ Ready for Marquez lineage tracking
✓ Ready for RAG pipeline integration
